In [ ]:
# -*- coding: utf-8 -*-
"""
RF-Temp (treinado com todas as temperaturas)
RF-Comp (treinado com TEMPS_TREINO e testado em TEMPS_PROVA)
vs Park
"""

import re, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore", category=UserWarning)

# ========= Parâmetros =========
REF_TEMP      = 20
FREQ_MIN_KHZ  = 30
FREQ_MAX_KHZ  = 50
PKL_TREINO    = "base_treino.pkl"
PKL_PROVA     = "base_prova (1).pkl"

TEMPS_TREINO = {0, 10, 20, 40, 60}
TEMPS_PROVA  = {-10, 30, 50, 70}

N_BANDS       = 4
PLOT_N_EXAMPLES = 6

RF_COMP_PARAMS = dict(
    n_estimators=600,
    max_depth=None,
    min_samples_leaf=2,
    min_samples_split=4,
    max_features="sqrt",
    bootstrap=True,
    n_jobs=-1,
    random_state=42,
)
RF_TEMP_PARAMS = dict(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    max_features="sqrt",
    bootstrap=True,
    n_jobs=-1,
    random_state=7,
)

# ========= Helpers =========
def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None

def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None and fmin_khz <= f/1e3 <= fmax_khz:
            cols.append(c); freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs, float)[order]

def rmsd(y_ref, y):   return float(np.sqrt(np.mean((y_ref - y)**2)))
def ccdm(y_ref, y):
    y1, y2 = y_ref - y_ref.mean(), y - y.mean()
    den = (np.linalg.norm(y1)*np.linalg.norm(y2))+1e-12
    rho = float(np.clip(np.dot(y1, y2)/den, -1, 1))
    return 1.0 - rho

def corr_per_sample(Y, Yhat):
    out=[]
    for i in range(Y.shape[0]):
        y,yh=Y[i],Yhat[i]
        num=((y-y.mean())*(yh-yh.mean())).sum()
        den=np.sqrt(((y-y.mean())**2).sum()*((yh-yh.mean())**2).sum())+1e-12
        out.append(float(np.clip(num/den,-1,1)))
    return np.array(out)

def sam_per_sample(Y,Yhat):
    out=[]
    for i in range(Y.shape[0]):
        y,yh=Y[i],Yhat[i]
        den=(np.linalg.norm(y)*np.linalg.norm(yh))+1e-12
        cosang=float(np.clip(np.dot(y,yh)/den,-1,1))
        out.append(float(np.degrees(np.arccos(cosang))))
    return np.array(out)

def nrmse_per_sample(Y,Yhat):
    out=[]
    for i in range(Y.shape[0]):
        y,yh=Y[i],Yhat[i]
        rmse=np.sqrt(np.mean((y-yh)**2))
        rng=np.max(y)-np.min(y)
        out.append(float(rmse/(rng+1e-12)))
    return np.array(out)

def eval_all_metrics(Y_true, Y_pred):
    y1, y2 = Y_true.reshape(-1), Y_pred.reshape(-1)
    return dict(
        R2   = r2_score(y1, y2),
        RMSE = float(np.sqrt(mean_squared_error(y1, y2))),
        MAE  = float(mean_absolute_error(y1, y2)),
        Corr = float(corr_per_sample(Y_true, Y_pred).mean()),
        SAM_deg = float(sam_per_sample(Y_true, Y_pred).mean()),
        NRMSE = float(nrmse_per_sample(Y_true, Y_pred).mean()),
        RMSD = float(np.mean([rmsd(Y_true[i], Y_pred[i]) for i in range(Y_true.shape[0])])),
        CCDM = float(np.mean([ccdm(Y_true[i], Y_pred[i]) for i in range(Y_true.shape[0])])),
    )

def print_metrics_block(title, m):
    keys = ["R2","RMSE","MAE","Corr","SAM_deg","NRMSE","RMSD","CCDM"]
    print(f"{title}: " + " | ".join([f"{k}={m[k]:.4f}" for k in keys]))

def add_extra_features(X):
    mu  = X.mean(axis=1, keepdims=True)
    sd  = X.std(axis=1,  keepdims=True)
    amp = (X.max(axis=1)-X.min(axis=1)).reshape(-1,1)
    return np.hstack([X, mu, sd, amp])

def add_temp_feature(X_aug, temp_vec):
    return np.hstack([X_aug, np.asarray(temp_vec).reshape(-1,1).astype(float)])

# ========= Park =========
def park_va_for_shift(z_ref, z, k):
    n = len(z_ref)
    if k >= 0:
        i0,i1,j0,j1 = 0, n-k, k, n
    else:
        i0,i1,j0,j1 = -k, n, 0, n+k
    if i1<=i0 or j1<=j0:
        return np.inf, 0.0, None
    zr = z_ref[i0:i1]; zd = z[j0:j1]
    delta_s = float((zr - zd).mean())
    diff = zr - (zd + delta_s)
    Va = float(np.sum(diff*diff))
    z_shift = np.full_like(z_ref, np.nan, dtype=float)
    z_shift[i0:i1] = zd + delta_s
    return Va, delta_s, z_shift

def park_compensate_curve(z_ref, z, max_shift=None):
    n = len(z_ref)
    if max_shift is None:
        max_shift = max(1, n//10)
    best = (np.inf, 0.0, None)
    for k in range(-max_shift, max_shift+1):
        Va, ds, zc = park_va_for_shift(z_ref, z, k)
        if Va < best[0]:
            best = (Va, ds, zc)
    _, _, zc = best
    if np.isnan(zc).any():
        zc = zc.copy()
        idx_valid = np.where(~np.isnan(zc))[0]
        if len(idx_valid)>0:
            first,last = idx_valid[0], idx_valid[-1]
            zc[:first] = zc[first]
            zc[last+1:] = zc[last]
        else:
            zc = z_ref.copy()
    return zc

def park_compensate_batch(y_ref, X):
    return np.vstack([park_compensate_curve(y_ref, X[i]) for i in range(X.shape[0])])

# ========= Carrega bases =========
base_tr = pd.read_pickle(PKL_TREINO)
base_te = pd.read_pickle(PKL_PROVA)
base_all = pd.concat([base_tr, base_te], ignore_index=True)

# ========= Frequências =========
freq_cols, _ = get_freq_columns(base_all, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
fhz = np.array([extract_freq_hz(c) for c in freq_cols], float)

# ========= Referência real @20°C =========
pool_20 = base_all.loc[base_all["temp_c"]==REF_TEMP, freq_cols].to_numpy(float)
assert len(pool_20)>0, "Não há curva real @20°C na base!"
y_ref = np.median(pool_20, axis=0)

# ========= RF-Temp (treinado com tudo) =========
X_temp = base_all[freq_cols].to_numpy(float)
X_temp_aug = add_extra_features(X_temp)
T_all = base_all["temp_c"].to_numpy(float)

rf_temp = RandomForestRegressor(**RF_TEMP_PARAMS)
rf_temp.fit(X_temp_aug, T_all)
T_hat_all = rf_temp.predict(X_temp_aug)

print("\n== RF-Temp (todas as temperaturas) ==")
print(f"Correlação: {np.corrcoef(T_all, T_hat_all)[0,1]:.4f} | RMSE={np.sqrt(mean_squared_error(T_all, T_hat_all)):.4f} | MAE={mean_absolute_error(T_all, T_hat_all):.4f}")

# ========= Conjuntos restritos p/ RF-Comp =========
tr_restr = base_all[base_all["temp_c"].isin(TEMPS_TREINO)].copy()
te_restr = base_all[base_all["temp_c"].isin(TEMPS_PROVA)].copy()

X_tr = tr_restr[freq_cols].to_numpy(float)
X_te = te_restr[freq_cols].to_numpy(float)

T_tr = tr_restr["temp_c"].to_numpy(float)
T_te_hat = rf_temp.predict(add_extra_features(X_te))  # usa T prevista em teste

# ========= RF-Comp =========
Y_tr_target = (y_ref[None,:] - X_tr)
X_tr_aug = add_extra_features(X_tr)
X_tr_comp = add_temp_feature(X_tr_aug, T_tr)

rf_comp = RandomForestRegressor(**RF_COMP_PARAMS)
t0=time.time()
rf_comp.fit(X_tr_comp, Y_tr_target)
print(f"[INFO] Tempo RF-Comp (treino restrito): {time.time()-t0:.1f}s")

# --- predições ---
X_te_aug = add_extra_features(X_te)
X_te_comp = add_temp_feature(X_te_aug, T_te_hat)
Y_te_hat = X_te + rf_comp.predict(X_te_comp)
Y_te_park = park_compensate_batch(y_ref, X_te)
Y_ref_te = np.tile(y_ref, (Y_te_hat.shape[0], 1))

# ========= Métricas globais =========
print("\n== MÉTRICAS (teste restrito) ==")
print_metrics_block("RF-Comp (prova)", eval_all_metrics(Y_ref_te, Y_te_hat))
print_metrics_block("Park    (prova)", eval_all_metrics(Y_ref_te, Y_te_park))

# ========= Métricas por banda =========
def band_indices(n, n_bands): return [np.array(ix,int) for ix in np.array_split(np.arange(n), n_bands)]
idx_bands = band_indices(Y_ref_te.shape[1], N_BANDS)
print("\n== MÉTRICAS por banda ==")
for b, ix in enumerate(idx_bands):
    mr = eval_all_metrics(Y_ref_te[:,ix], Y_te_hat[:,ix])
    mp = eval_all_metrics(Y_ref_te[:,ix], Y_te_park[:,ix])
    print(f"Banda {b}: RF-Comp -> R2={mr['R2']:.3f} | RMSE={mr['RMSE']:.3f} | MAE={mr['MAE']:.3f} | RMSD={mr['RMSD']:.3f} | CCDM={mr['CCDM']:.3f}")
    print(f"          Park    -> R2={mp['R2']:.3f} | RMSE={mp['RMSE']:.3f} | MAE={mp['MAE']:.3f} | RMSD={mp['RMSD']:.3f} | CCDM={mp['CCDM']:.3f}")

# ========= Plots =========
def pick_examples(base_df, n=PLOT_N_EXAMPLES):
    temps = sorted(base_df["temp_c"].unique())
    idxs=[]
    for T in temps:
        cand = np.where(base_df["temp_c"].to_numpy(float)==T)[0]
        if len(cand): idxs.append(int(cand[0]))
    rest = [i for i in range(len(base_df)) if i not in idxs]
    idxs += rest[:max(0, n-len(idxs))]
    return idxs[:n]

def plot_examples(fhz, y_ref, X_orig, Y_rf, Y_pk, base_df, n=PLOT_N_EXAMPLES):
    fhz_khz = fhz/1e3
    idxs = pick_examples(base_df, n)
    for i in idxs:
        plt.figure(figsize=(8,4.5))
        plt.plot(fhz_khz, X_orig[i], label=f"Original @ {base_df.iloc[i]['temp_c']}°C", lw=1.2)
        plt.plot(fhz_khz, y_ref,     label=f"Referência @ {REF_TEMP}°C", lw=1.8)
        plt.plot(fhz_khz, Y_rf[i],   label="RF-Comp → 20°C", lw=1.5)
        plt.plot(fhz_khz, Y_pk[i],   label="Park → 20°C", lw=1.5)
        plt.xlabel("Frequência (kHz)"); plt.ylabel("Impedância (un.)")
        plt.title(f"Amostra {i} — Compensação para {REF_TEMP}°C")
        plt.grid(alpha=0.25); plt.legend(); plt.tight_layout()
    plt.show()

print("\n== PLOTS de exemplos ==")
plot_examples(fhz, y_ref, X_te, Y_te_hat, Y_te_park, te_restr, n=PLOT_N_EXAMPLES)

print("\n[FIM]")
